# Mecanismos de atención

En la práctica anterior convertimos texto en **embeddings de entrada**. En esta práctica implementaremos el corazón de un LLM tipo GPT: el **mecanismo de atención**, que transforma cada embedding en un **vector de contexto** que combina información de todos los tokens de la secuencia.

Construiremos el mecanismo en cuatro etapas, cada una sobre la anterior:

1. **Atención simplificada**, sin pesos entrenables, para entender la idea.
2. **Self-attention** con pesos entrenables (*scaled dot-product attention*).
3. **Atención causal**, que impide ver los tokens futuros, más *dropout*.
4. **Atención multi-cabeza** (*multi-head attention*), la versión que usa GPT-2.

La clase `MultiHeadAttention` que programes al final es la que usaremos en la próxima lección dentro del bloque *transformer* de nuestro GPT.

Basado en el capítulo 3 de *Build a Large Language Model (From Scratch)*, S. Raschka.

**Total de puntos: 6**. Cada sección marcada con `TODO` vale **1 punto**. Después de cada una hay una celda de **prueba** que verifica tu código y, si es correcto, te asigna el punto. Al final del notebook verás tu calificación.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irvingvasquez/practicas_llm_vlm/blob/main/03_01_mecanismos_atencion.ipynb)

@juan1rving

In [ ]:
# En Colab descargamos el repositorio para tener el paquete cllm y los datos
import os, sys
if "google.colab" in sys.modules and not os.path.exists("cllm"):
    !git clone -q https://github.com/irvingvasquez/practicas_llm_vlm.git
    %cd practicas_llm_vlm
!pip install -q tiktoken

In [ ]:
# Primero llamamos a los paquetes necesarios
from importlib.metadata import version

import torch
from torch import nn
import tiktoken

from cllm import calificar, resumen, cargar_en_paz
from cllm.pruebas_03_01 import prueba_1, prueba_2, prueba_3, prueba_4, prueba_5, prueba_6

print("torch version:", version("torch"))

## 1. ¿Por qué atención?

Antes de los *transformers*, las redes recurrentes (RNN) procesaban el texto palabra por palabra y debían comprimir toda la oración en un único estado oculto, lo que dificultaba recordar información lejana. Los mecanismos de atención resuelven esto permitiendo que, al procesar cada token, el modelo **consulte directamente** a todos los demás tokens y decida cuánto le importa cada uno.

En **self-attention**, la secuencia se atiende a sí misma: para cada token de entrada $x^{(i)}$ calculamos un **vector de contexto** $z^{(i)}$, que es una versión enriquecida del embedding que incorpora información de toda la secuencia.

Trabajaremos con la oración *"Tu viaje empieza con un paso"*, ya convertida en embeddings de 3 dimensiones (una dimensión pequeña para poder leer los números):

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Tu       (x^1)
   [0.55, 0.87, 0.66], # viaje    (x^2)
   [0.57, 0.85, 0.64], # empieza  (x^3)
   [0.22, 0.58, 0.33], # con      (x^4)
   [0.77, 0.25, 0.10], # un       (x^5)
   [0.05, 0.80, 0.55]] # paso     (x^6)
)

## 2. Atención simplificada, sin pesos entrenables

Calculemos el vector de contexto $z^{(2)}$ del segundo token (*viaje*), que llamaremos la **consulta** (*query*). Son tres pasos:

1. **Puntajes de atención** $\omega_{2j}$: el producto punto entre la consulta $x^{(2)}$ y cada entrada $x^{(j)}$. El producto punto mide qué tan alineados (similares) están dos vectores.
2. **Pesos de atención** $\alpha_{2j}$: normalizamos los puntajes con **softmax** para que sean positivos y sumen 1.
3. **Vector de contexto**: $z^{(2)} = \sum_j \alpha_{2j}\, x^{(j)}$, la suma de las entradas ponderada por los pesos.

In [ ]:
query = inputs[1]  # la segunda entrada es la consulta

# Paso 1: puntajes de atención
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print("Puntajes:", attn_scores_2)

# Paso 2: pesos de atención (softmax)
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Pesos:   ", attn_weights_2, "  suma =", attn_weights_2.sum().item())

# Paso 3: vector de contexto
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print("Contexto z(2):", context_vec_2)

Usamos `torch.softmax` en lugar de dividir entre la suma porque maneja mejor los valores extremos, garantiza pesos positivos y tiene mejores propiedades para el gradiente. Además es numéricamente más estable que una implementación ingenua `torch.exp(x) / torch.exp(x).sum()`.

### Para todos los tokens a la vez

Los ciclos `for` son lentos. Si colocamos las entradas como filas de una matriz $X$ de forma `(num_tokens, d)`, los tres pasos para **todos** los tokens se escriben con multiplicaciones de matrices:

1. Puntajes: $\Omega = X X^\top$, de forma `(num_tokens, num_tokens)`. La fila $i$ contiene los puntajes de la consulta $i$.
2. Pesos: softmax **sobre cada fila** (la última dimensión, `dim=-1`).
3. Contextos: $Z = A X$, donde $A$ es la matriz de pesos.

En PyTorch, `@` es la multiplicación de matrices y `.T` la transpuesta.

In [ ]:
def atencion_simple(x):
    # x: tensor de forma (num_tokens, d)
    # Regresa (contexto, pesos):
    #   contexto: (num_tokens, d), un vector de contexto por token
    #   pesos:    (num_tokens, num_tokens), cada fila suma 1
    # TODO: implementa los tres pasos con multiplicaciones de matrices, sin ciclos for.
    # 1 punto
    pesos = None
    contexto = None

    return contexto, pesos

In [ ]:
# Prueba del punto 1
calificar(1, prueba_1, atencion_simple)

In [ ]:
all_context_vecs, attn_weights = atencion_simple(inputs)
print(attn_weights)
print(all_context_vecs)
print("¿La fila 2 coincide con z(2)?", torch.allclose(all_context_vecs[1], context_vec_2))

## 3. Self-attention con pesos entrenables

En la atención simplificada, los pesos dependen sólo de los embeddings: no hay nada que aprender. La **self-attention** que usan GPT y el *transformer* original agrega tres matrices de pesos entrenables $W_q$, $W_k$ y $W_v$ que proyectan cada entrada en tres vectores:

- **Consulta (query)** $q = x W_q$: lo que el token actual "busca".
- **Llave (key)** $k = x W_k$: con qué se compara cada token para decidir su relevancia.
- **Valor (value)** $v = x W_v$: la información que cada token aporta al contexto.

La analogía es una base de datos: la consulta se compara con las llaves y se recuperan los valores correspondientes. Ahora:

$$\text{puntajes} = Q K^\top, \qquad A = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right), \qquad Z = A V$$

Dividimos entre $\sqrt{d_k}$ (la raíz de la dimensión de las llaves) porque con dimensiones grandes los productos punto crecen mucho, el softmax se vuelve casi un *one-hot* y los gradientes se hacen muy pequeños. Por eso se llama ***scaled* dot-product attention**.

En GPT la dimensión de entrada y salida es la misma, pero aquí usaremos `d_in = 3` y `d_out = 2` para seguir los cálculos con facilidad.

In [ ]:
x_2 = inputs[1]
d_in = inputs.shape[1]  # 3
d_out = 2

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
keys = inputs @ W_key
values = inputs @ W_value
print("query_2:", query_2)
print("keys:", keys.shape, " values:", values.shape)

attn_scores_2 = query_2 @ keys.T
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
context_vec_2 = attn_weights_2 @ values
print("Pesos:", attn_weights_2)
print("Contexto z(2):", context_vec_2)

Organicemos este cálculo en un módulo de PyTorch para todos los tokens. La siguiente clase (ya implementada) guarda las tres matrices como `nn.Parameter`:

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T  # omega
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

Es mejor usar capas `nn.Linear` sin sesgo (`bias=False`): hacen la misma multiplicación de matrices pero tienen un esquema de **inicialización de pesos** más adecuado, lo que hace el entrenamiento más estable. Además permiten agregar sesgo opcional con `qkv_bias=True`.

Implementa `SelfAttention_v2`:

- En `__init__`, crea `self.W_query`, `self.W_key` y `self.W_value` como `nn.Linear(d_in, d_out, bias=qkv_bias)`. Usa **exactamente** esos nombres.
- En `forward`, calcula las consultas, llaves y valores aplicando las capas a `x`, y después los puntajes, los pesos escalados por $\sqrt{d_k}$ con softmax en la última dimensión, y los vectores de contexto.

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # TODO: define las tres capas lineales
        # 0.5 puntos
        self.W_query = None
        self.W_key = None
        self.W_value = None

    def forward(self, x):
        # TODO: calcula los vectores de contexto con scaled dot-product attention
        # 0.5 puntos
        context_vec = None

        return context_vec

In [ ]:
# Prueba del punto 2
calificar(2, prueba_2, SelfAttention_v2)

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

### Pesos de `nn.Linear` vs. `nn.Parameter`

`SelfAttention_v1` y `SelfAttention_v2` dan resultados distintos sólo porque sus pesos iniciales son diferentes. Si copiamos los pesos de una a la otra deben dar **exactamente** lo mismo.

El detalle es que `nn.Linear(d_in, d_out)` guarda su matriz `weight` con forma **`(d_out, d_in)`** y calcula $y = x W^\top$, mientras que en `SelfAttention_v1` las matrices tienen forma `(d_in, d_out)` y se calcula $y = x W$. Es decir, una es la **transpuesta** de la otra.

Este detalle será muy importante cuando, al final del curso, carguemos en nuestro modelo los pesos preentrenados de GPT-2 publicados por OpenAI: tendremos que acomodar matrices de un formato a otro.

Implementa `transferir_pesos(sa_v2, sa_v1)`, que copia las tres matrices de `sa_v2` a `sa_v1`. Recuerda que los atributos de `sa_v1` deben seguir siendo `nn.Parameter`; por ejemplo, puedes asignar `nn.Parameter(...)` o modificar el tensor dentro de `with torch.no_grad():`.

In [ ]:
def transferir_pesos(sa_v2, sa_v1):
    # Copia los pesos de las capas nn.Linear de sa_v2 a los nn.Parameter de sa_v1.
    # TODO
    # 1 punto
    pass

In [ ]:
# Prueba del punto 3
calificar(3, prueba_3, transferir_pesos, SelfAttention_v1)

In [ ]:
transferir_pesos(sa_v2, sa_v1)
print("v1:\n", sa_v1(inputs))
print("v2:\n", sa_v2(inputs))

## 4. Atención causal: ocultar las palabras futuras

Un LLM tipo GPT genera texto **de izquierda a derecha**: para predecir el siguiente token sólo puede usar los tokens anteriores. Por eso usamos **atención causal** (o enmascarada): al calcular el contexto del token $i$, los pesos de atención de los tokens $j > i$ deben ser **cero**.

Una primera idea es aplicar softmax y luego poner en cero lo que está arriba de la diagonal con `torch.tril`, volviendo a normalizar cada fila:

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

masked_simple = attn_weights * mask_simple
masked_simple_norm = masked_simple / masked_simple.sum(dim=-1, keepdim=True)
print(masked_simple_norm)

Hay una forma más eficiente que da el mismo resultado: **antes** del softmax, sustituimos por $-\infty$ los puntajes de arriba de la diagonal. Como $e^{-\infty} = 0$, el softmax les asigna peso cero y las filas siguen sumando 1. Para ello se crea una máscara con `torch.triu(..., diagonal=1)` (unos arriba de la diagonal) y se usa `masked_fill`:

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

### Dropout en la atención

Para reducir el sobreajuste, GPT aplica **dropout** a la matriz de pesos de atención: durante el entrenamiento pone en cero una fracción aleatoria de los pesos y escala los restantes por $1/(1-p)$ para conservar el promedio. En modo evaluación (`model.eval()`) el dropout no hace nada.

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
print(dropout(torch.ones(6, 6)))
print(dropout(attn_weights))

### Una clase de atención causal

Ahora juntaremos todo en la clase `CausalAttention`, que además debe procesar **lotes**: la entrada tendrá forma `(batch, num_tokens, d_in)`, como los que produce el `DataLoader` de la práctica anterior. Simulamos un lote duplicando la oración:

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  # (2 textos, 6 tokens, 3 dimensiones)

Implementa `CausalAttention`:

- En `__init__`:
  - Las capas `W_query`, `W_key` y `W_value` igual que en `SelfAttention_v2`.
  - `self.dropout = nn.Dropout(dropout)`.
  - La máscara de tamaño `(context_length, context_length)` con unos arriba de la diagonal, registrada con `self.register_buffer('mask', ...)`. Un *buffer* no es un parámetro entrenable, pero se mueve con el modelo al GPU (`model.to(device)`) y se guarda junto con sus pesos.
- En `forward(x)`:
  - Con lotes, `keys.T` ya no sirve: hay que transponer sólo las dos últimas dimensiones, `keys.transpose(1, 2)`.
  - El texto puede ser más corto que `context_length`, así que usa sólo la parte necesaria de la máscara: `self.mask.bool()[:num_tokens, :num_tokens]`.
  - Aplica la máscara con $-\infty$, luego el softmax escalado, después el dropout a los pesos y finalmente calcula los vectores de contexto.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        # TODO: capas W_query, W_key, W_value, dropout y la máscara como buffer
        # 0.5 puntos

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # TODO: atención causal con dropout
        # 0.5 puntos
        context_vec = None

        return context_vec

In [ ]:
# Prueba del punto 4
calificar(4, prueba_4, CausalAttention)

In [ ]:
torch.manual_seed(123)
ca = CausalAttention(d_in, d_out, context_length=batch.shape[1], dropout=0.0)
context_vecs = ca(batch)
print(context_vecs)
print("Forma:", context_vecs.shape)

## 5. Atención multi-cabeza

Una sola cabeza de atención aprende un único patrón de relaciones entre tokens. La **atención multi-cabeza** ejecuta varias cabezas en paralelo, cada una con sus propias matrices $W_q$, $W_k$ y $W_v$, para que cada una pueda especializarse (por ejemplo, una en relaciones sintácticas y otra en la palabra anterior). Sus vectores de contexto se **concatenan**.

La forma más directa de implementarla es apilar varias `CausalAttention`. Implementa `MultiHeadAttentionWrapper`:

- En `__init__`, crea `self.heads` como un `nn.ModuleList` con `num_heads` instancias de tu `CausalAttention`. Es importante usar `nn.ModuleList` y no una lista de Python: así PyTorch registra los parámetros de cada cabeza y se entrenan y mueven al GPU con el modelo.
- En `forward`, aplica cada cabeza a `x` y concatena los resultados en la **última** dimensión con `torch.cat`. Con `num_heads` cabezas de dimensión `d_out`, la salida tendrá dimensión `num_heads * d_out`.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # TODO: crea las cabezas
        # 0.5 puntos
        self.heads = None

    def forward(self, x):
        # TODO: aplica cada cabeza y concatena
        # 0.5 puntos
        return None

In [ ]:
# Prueba del punto 5
calificar(5, prueba_5, MultiHeadAttentionWrapper)

In [ ]:
torch.manual_seed(123)
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length=batch.shape[1], dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("Forma:", context_vecs.shape)

**Ejercicio (sin puntos).** Cambia los argumentos de `MultiHeadAttentionWrapper(..., num_heads=2)` para que los vectores de contexto de salida tengan 2 dimensiones en lugar de 4, sin cambiar `num_heads` ni la clase.

In [ ]:
# Escribe aquí tu código

### Atención multi-cabeza eficiente

El *wrapper* procesa las cabezas **una tras otra**. Es más eficiente calcularlas todas a la vez: usamos **una sola** capa `W_query` de tamaño `d_in → d_out` y después **dividimos** su salida en `num_heads` cabezas de dimensión `head_dim = d_out // num_heads`. Lo mismo para las llaves y los valores. Matemáticamente es lo mismo que el *wrapper*, pero con una sola multiplicación de matrices grande en lugar de varias pequeñas.

La división se hace cambiando la forma de los tensores. Sigue este recorrido de formas (`b` = lote, `T` = tokens, `h` = cabezas, `hd` = `head_dim`):

| Paso | Operación | Forma resultante |
|---|---|---|
| 1 | `queries = self.W_query(x)` (igual para llaves y valores) | `(b, T, d_out)` |
| 2 | `.view(b, T, h, hd)`: separa las cabezas | `(b, T, h, hd)` |
| 3 | `.transpose(1, 2)`: agrupa por cabeza | `(b, h, T, hd)` |
| 4 | `queries @ keys.transpose(2, 3)`: puntajes de cada cabeza | `(b, h, T, T)` |
| 5 | máscara $-\infty$, softmax escalado por $\sqrt{hd}$, dropout | `(b, h, T, T)` |
| 6 | `attn_weights @ values` | `(b, h, T, hd)` |
| 7 | `.transpose(1, 2)`: vuelve a poner los tokens primero | `(b, T, h, hd)` |
| 8 | `.contiguous().view(b, T, d_out)`: concatena las cabezas | `(b, T, d_out)` |
| 9 | `self.out_proj(...)`: combina la información de las cabezas | `(b, T, d_out)` |

Dos detalles importantes:

- En el paso 2 **no** puedes hacer directamente `.view(b, h, T, hd)`: eso mezclaría tokens de distintas posiciones dentro de una misma cabeza. Primero se separa la última dimensión y después se transpone.
- `transpose` no reacomoda la memoria del tensor, por eso hace falta `.contiguous()` antes del `.view` del paso 8.

Además, la clase agrega una **proyección de salida** `out_proj = nn.Linear(d_out, d_out)` (con sesgo), como en GPT-2.

Implementa `MultiHeadAttention`:

- En `__init__`:
  - Verifica con `assert` que `d_out` sea divisible entre `num_heads`.
  - Guarda `self.d_out`, `self.num_heads` y `self.head_dim`.
  - Crea `W_query`, `W_key`, `W_value` (de `d_in` a `d_out`, con `bias=qkv_bias`), `out_proj`, `dropout` y el buffer `mask`.
- En `forward`, sigue la tabla.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # TODO: atributos, capas y máscara
        # 0.5 puntos

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # TODO: sigue los 9 pasos de la tabla
        # 0.5 puntos
        context_vec = None

        return context_vec

In [ ]:
# Prueba del punto 6
calificar(6, prueba_6, MultiHeadAttention)

In [ ]:
torch.manual_seed(123)
mha = MultiHeadAttention(d_in, d_out, context_length=batch.shape[1], dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("Forma:", context_vecs.shape)

## 6. Atención con el tamaño de GPT-2

El GPT-2 más pequeño (124M parámetros) usa embeddings de **768** dimensiones, **12** cabezas de atención y un contexto de **1024** tokens; en GPT la dimensión de entrada y salida de la atención es la misma (`d_in = d_out`).

Juntemos esta práctica con la anterior: tokenizamos el poema *En paz* con el BPE de GPT-2, lo convertimos en embeddings de entrada (tokens + posiciones) y lo pasamos por una capa de atención multi-cabeza de tamaño GPT-2.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # tamaño del vocabulario BPE
    "context_length": 1024,  # longitud máxima de contexto
    "emb_dim": 768,          # dimensión de los embeddings
    "n_heads": 12,           # número de cabezas de atención
    "drop_rate": 0.1,        # tasa de dropout
    "qkv_bias": False,       # sesgo en las capas de consulta, llave y valor
}
cfg = GPT_CONFIG_124M

# Texto -> IDs: dos ventanas de 8 tokens
tokenizer = tiktoken.get_encoding("gpt2")
ids = tokenizer.encode(cargar_en_paz())
inputs_ids = torch.tensor([ids[0:8], ids[8:16]])
print("IDs:", inputs_ids.shape)

# IDs -> embeddings de entrada (lo que implementaste en la práctica anterior)
tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
x = tok_emb(inputs_ids) + pos_emb(torch.arange(inputs_ids.shape[1]))
print("Embeddings de entrada:", x.shape)

# Embeddings -> vectores de contexto
mha_gpt2 = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                              cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
salida = mha_gpt2(x)
print("Vectores de contexto:", salida.shape)

n_param = sum(p.numel() for p in mha_gpt2.parameters())
print(f"Parámetros en una capa de atención de GPT-2 small: {n_param:,}")
print(f"En sus 12 bloques transformer: {12 * n_param:,}")

## Calificación

In [ ]:
resumen(6)

## Resumen

- La **atención** transforma cada embedding en un **vector de contexto**: una suma de la información de toda la secuencia, ponderada por su relevancia.
- En la versión simplificada, los pesos son el softmax de los productos punto entre las entradas; las multiplicaciones de matrices sustituyen a los ciclos.
- La ***scaled dot-product attention*** agrega matrices entrenables para consultas, llaves y valores, y escala los puntajes por $\sqrt{d_k}$.
- `nn.Linear` guarda sus pesos **transpuestos**, `(d_out, d_in)`.
- La **máscara causal** impide que un token vea a los futuros, y el **dropout** en los pesos de atención reduce el sobreajuste.
- La **atención multi-cabeza** ejecuta varias atenciones en paralelo; la versión eficiente las calcula con una sola proyección y cambios de forma (`view` y `transpose`).

En la siguiente lección usaremos `MultiHeadAttention` para construir el **bloque transformer** y la arquitectura completa de GPT-2.